# SPOA: VG150 Standardized Benchmark

This notebook trains SPOA (Subject+Attribute → Predicate → Object+Attribute) specifically for the **VG150** benchmark (150 objects, 50 predicates).

Key points:
- Uses VG150 standardized dataset and vocabulary
- Uses 1:1 loss weighting between SPOA alignment and Dense Grounding
- Default dataset: `anhkhoa1804/VG150-SGG-Standard` (Hugging Face)
- Recommended for Kaggle: use `--stage 1` with `--vg150_enabled True` (warmup on T4)
- Scale to Stage 2 on 2x GPUs or Stage 3 on 8x GPUs for production

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q datasets huggingface_hub spacy transformers ftfy torch torchvision
!python -m spacy download en_core_web_sm

import torch
import os
import sys
from pathlib import Path

# Verify GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Clone Repository & Setup

In [ ]:
import os

# Minimal environment setup for Kaggle: set HF dataset ID and optional token
os.environ['HF_DATASET_ID'] = "anhkhoa1804/VG150-SGG-Standard"
# If you have an HF token, set it in the UI secrets or uncomment next line:
# os.environ['HF_TOKEN'] = "<YOUR_HF_TOKEN>"

print("Environment configured for VG150 streaming.")

## Step 3: Verify SPOA Dataset on Hub

In [ ]:
from datasets import load_dataset

# VG150 dataset ID (standardized VG150 SGG)
DATASET_ID = "anhkhoa1804/VG150-SGG-Standard"

print(f"Using VG150 dataset: {DATASET_ID} (streaming)")

try:
    ds = load_dataset(DATASET_ID, split="train", streaming=True)
    sample = next(iter(ds))
    print(f"Sample keys: {list(sample.keys())}")
except Exception as e:
    print(f"Error loading VG150 streaming dataset: {e}")
    print("Ensure HF token and dataset availability.")

## Step 4: Setup Training Environment

In [ ]:
# Setup environment variables for training
os.environ["HF_DATASET_ID"] = DATASET_ID
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OMP_NUM_THREADS"] = "1"

# Create output directories
WORKING_DIR = Path("/kaggle/working")
CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
RUNS_DIR = WORKING_DIR / "runs"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Output directories created:")
print(f"  - Checkpoints: {CHECKPOINT_DIR}")
print(f"  - Runs/Logs: {RUNS_DIR}")

## Step 5: Launch SPOA Training (T4 Pilot Configuration)

In [ ]:
import subprocess
import os
import sys
from datetime import datetime
from pathlib import Path

# Setup environment (repo is already in working dir for Kaggle)
REPO_ROOT = Path("/kaggle/working")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

RUN_NAME = f"kaggle_t4_pilot_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_DIR = Path("/kaggle/working/runs") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "torchrun", "--nproc_per_node=1", "--master_port", "29500",
    "-m", "openvocab_rel.train",
    "--stage", "1",  # Stage 1: Warmup with frozen CLIP @ 336px
    "--run_name", RUN_NAME,
    "--out_dir", str(OUT_DIR),
    "--epochs", "1",
    "--hf_dataset_id", DATASET_ID,
    "--hf_train_split", "train",
    "--hf_val_split", "train",
    "--hf_streaming",
    "--vg150_enabled", "True",
    "--save_path", "/kaggle/working/checkpoints/kaggle_t4_spoa.pt",
    "--save_metrics_json", str(OUT_DIR / "metrics.jsonl")
]

print(f"Launching SPOA VG150 Training...")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()

## Step 6: Visualize Training Metrics

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

# Load metrics from JSONL
metrics_file = OUT_DIR / "metrics.jsonl"

if metrics_file.exists():
    print(f"Loading metrics from: {metrics_file}")
    
    # Read JSONL file
    records = []
    with open(metrics_file, 'r') as f:
        for line in f:
            records.append(json.loads(line))
    
    df = pd.DataFrame(records)
    print(f"\nLoaded {len(df)} metric records")
    print(f"Columns: {list(df.columns)}\n")
    print(df.head())
    
    # Plot loss curves if available
    if 'loss' in df.columns:
        plt.figure(figsize=(10, 5))
        plt.plot(df['loss'], label='Total Loss', marker='o')
        if 'loss_spoa' in df.columns:
            plt.plot(df['loss_spoa'], label='SPOA Loss', alpha=0.7)
        if 'loss_ground' in df.columns:
            plt.plot(df['loss_ground'], label='Grounding Loss', alpha=0.7)
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.title('SPOA Training Loss Curves (T4 Pilot)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(RUNS_DIR / f"{RUN_NAME}_loss_curves.png", dpi=100)
        plt.show()
        print("\n✓ Loss curves saved to:", RUNS_DIR / f"{RUN_NAME}_loss_curves.png")
else:
    print(f"⚠ Metrics file not found: {metrics_file}")

## Step 8: Summary & Artifacts

In [ ]:
import os

print("="*60)
print("SPOA T4 Pilot Training Complete!")
print("="*60)
print(f"\n📊 Run Name: {RUN_NAME}")
print(f"📁 Output Directory: {OUT_DIR}")
print(f"💾 Checkpoint: {CHECKPOINT_DIR / 'kaggle_t4_spoa.pt'}")
print(f"📈 Metrics: {OUT_DIR / 'metrics.jsonl'}")

# List artifacts
print(f"\n📦 Generated Files:")
for f in sorted(CHECKPOINT_DIR.glob("*")):
    size_mb = f.stat().st_size / 1e6
    print(f"  - {f.name} ({size_mb:.1f} MB)")

for f in sorted(OUT_DIR.glob("*")):
    size_mb = f.stat().st_size / 1e6
    print(f"  - {f.name} ({size_mb:.1f} MB)")

print(f"\n✨ All artifacts saved to /kaggle/working/")
print(f"\n🔗 Next Steps:")
print(f"  1. Download checkpoint and metrics")
print(f"  2. Verify dataset on HF Hub: https://huggingface.co/datasets/{DATASET_ID}")
print(f"  3. Scale training on H200 (Stage 3): bash scripts/run_h200.sh checkpoints/stage2_checkpoint.pt 5")